In [2]:
%conda init
%conda activate studiolab
%conda env list
%pip list

no change     /opt/conda/condabin/conda
no change     /opt/conda/bin/conda
no change     /opt/conda/bin/conda-env
no change     /opt/conda/bin/activate
no change     /opt/conda/bin/deactivate
no change     /opt/conda/etc/profile.d/conda.sh
no change     /opt/conda/etc/fish/conf.d/conda.fish
no change     /opt/conda/shell/condabin/Conda.psm1
no change     /opt/conda/shell/condabin/conda-hook.ps1
no change     /opt/conda/lib/python3.9/site-packages/xontrib/conda.xsh
no change     /opt/conda/etc/profile.d/conda.csh
no change     /home/studio-lab-user/.bashrc
No action taken.

Note: you may need to restart the kernel to use updated packages.

CommandNotFoundError: Your shell has not been properly configured to use 'conda activate'.
To initialize your shell, run

    $ conda init <SHELL_NAME>

Currently supported shells are:
  - bash
  - fish
  - tcsh
  - xonsh
  - zsh
  - powershell

See 'conda init --help' for more information and options.

IMPORTANT: You may need to close and restart you

In [2]:
import os
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
import gymnasium
from gymnasium.wrappers.normalize import NormalizeObservation
from gymnasium.wrappers.transform_observation import TransformObservation
import numpy as np
import shadow_gym # type: ignore

"""
Author: Ethan Cheam
Trains a PPO model, saves models at regular intervals, and record training performance in Tensorboard.
"""

# SETTINGS
# When starting from an existing model, run_name is name of original run and rerun_name is name of logs and models of the new run
start_from_existing = False
existing_model_file = os.path.join("PPO-17c-shadowgym=rerun-2","1000000") # no need .zip extension
re_run_name = "PPO-17c-shadowgym-rerun-3"

# Run name should have model, unique number, and optionally a description
run_name = "PPO-20-shadowgym-ethan"
saving_timesteps_interval = 25_000
start_saving = 300_000

# Set up folders to store models and logs
models_dir = os.path.join(os.path.dirname(__file__), 'models')
logs_dir = os.path.join(os.path.dirname(__file__), 'logs')

if start_from_existing:
    run_dir = os.path.join(models_dir, re_run_name)
    run_logs_dir = os.path.join(logs_dir, re_run_name)
else:
    run_dir = os.path.join(models_dir, run_name)
    run_logs_dir = os.path.join(logs_dir, run_name)

if os.path.exists(run_dir):
    raise Exception("Error: model folder already exists. Change run_name to prevent overriding existing model folder")
if os.path.exists(run_logs_dir):
    raise Exception("Error: log folder already exists. Change run_name to prevent overriding existing log folder")

def clip_observation(obs):
    """
    clips observation to within 5 standard deviations of the mean
    Refer to section D.1 of Open AI paper
    """
    return np.clip(obs,a_min=obs.mean() - (5 * obs.std()), a_max=obs.mean() + (5 * obs.std()))

def get_old_model(run_dir) -> str:
    """
    Returns full path of old model to delete. A model must end in .zip.
    Before first model is saved, returns "" indicating no existing model
    """
    if os.path.exists(run_dir):
        all_files = os.listdir(run_dir)
        if all_files == []:
            return ""
        file = os.listdir(run_dir)[0]
        return os.path.join(run_dir, file)
    return ""

env = gymnasium.make("ShadowEnv-v0", GUI=False)
env = NormalizeObservation(env)
env = TransformObservation(env,f=clip_observation)
env = Monitor(env)


if start_from_existing:
    previous_model_path = os.path.join(models_dir, existing_model_file)
    model = PPO.load(previous_model_path, env)
else:
    model = PPO(policy="MlpPolicy", 
                env=env,
                learning_rate=3e-4,
                tensorboard_log=logs_dir,
                normalize_advantage=True,
                gamma=0.998,
                gae_lambda=0.95,
                ent_coef=0.01,
                clip_range=0.2,
                verbose=0,)

timesteps = 0
if start_from_existing:
    # Override run_name for below argument to tb_log_name
    run_name = re_run_name
while True:
    model.learn(saving_timesteps_interval, tb_log_name=run_name, reset_num_timesteps=False)
    timesteps += saving_timesteps_interval
    if timesteps >= start_saving:
        old = get_old_model(run_dir)
        model.save(os.path.join(run_dir, str(timesteps)))
        if old != "":
            os.remove(old)


ModuleNotFoundError: No module named 'stable_baselines3'